In [0]:
# Pull the values from your 'sql-credentials' scope
jdbc_host = dbutils.secrets.get(scope="sql-credentials", key="host")
jdbc_password = dbutils.secrets.get(scope="sql-credentials", key="password")

# Note: When you print these, Databricks will show '[REDACTED]' for security
print(f"Host: {jdbc_host}") 

Host: [REDACTED]


In [0]:
dbutils.secrets.list("sql-credentials")

[SecretMetadata(key='database'),
 SecretMetadata(key='host'),
 SecretMetadata(key='password'),
 SecretMetadata(key='username')]

In [0]:
# Notebook: jdbc_to_landing
from pyspark.sql import functions as F

# ── Connection Config ──────────────────────────────────────────
jdbc_url = (
    "jdbc:sqlserver://"
    + dbutils.secrets.get(scope="sql-credentials", key="host")
    + ";databaseName="
    + dbutils.secrets.get(scope="sql-credentials", key="database")
    + ";encrypt=true;trustServerCertificate=false;"
    + "hostNameInCertificate=*.database.windows.net;loginTimeout=30"
)

jdbc_properties = {
    "user"    : dbutils.secrets.get(scope="sql-credentials", key="username"),
    "password": dbutils.secrets.get(scope="sql-credentials", key="password"),
    "driver"  : "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# ── Config ─────────────────────────────────────────────────────
CATALOG    = "dbx_ws_project"       # e.g. "main"
SCHEMA     = "01_bronze"        # e.g. "default"
TABLES     = [
    "customers",
    "historical_orders",
    "restaurants",
    "reviews",
    "menu_items"
]

# ── Ingest Each Table ──────────────────────────────────────────
for table in TABLES:
    print(f"Ingesting: {table}...")

    df = (
        spark.read.jdbc(
            url        = jdbc_url,
            table      = f"dbo.{table}",
            properties = jdbc_properties
        )
        .withColumn("_load_timestamp", F.current_timestamp())  # CDC substitute
        .withColumn("_source_table",   F.lit(table))           # helpful metadata
    )

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")
    )

    print(f"✓ {table} → {table}/ ({df.count()} rows)")

print("\n✅ All tables successfully landed!")

Ingesting: customers...
✓ customers → customers/ (500 rows)
Ingesting: historical_orders...
✓ historical_orders → historical_orders/ (8000 rows)
Ingesting: restaurants...
✓ restaurants → restaurants/ (5 rows)
Ingesting: reviews...
✓ reviews → reviews/ (77 rows)
Ingesting: menu_items...
✓ menu_items → menu_items/ (145 rows)

✅ All tables successfully landed!
